# 04. 피처 엔지니어링 및 모델 입력 컬럼 확정

이 노트북은 `03_preprocess_windows.ipynb`의 산출물인 `data/processed/ml_windows/ml_window_dataset.csv`를 읽어 모델별 입력 컬럼 계약을 확정한다.

- `00`부터 `03`까지의 파일은 수정하지 않는다.
- Isolation Forest는 handoff metadata의 195개 feature를 상한으로 고정한다.
- 04에서는 새 파생 feature를 만들지 않는다.
- risk와 leadtime feature는 현재 03 산출물에 실제 존재하는 컬럼 기준으로 조정한다.
- 선택/제외 이유는 CSV 리포트와 `PREPROCESSING/docs/04_feature_engineering.md`에 남긴다.

In [1]:
from pathlib import Path
import json
import re

import pandas as pd

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists() and (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다. C:/project3.0_2 안에서 노트북을 실행하세요.")


ROOT = find_project_root(Path.cwd().resolve())
WINDOW_DATASET_PATH = ROOT / "data" / "processed" / "ml_windows" / "ml_window_dataset.csv"
OUT_DIR = ROOT / "data" / "processed" / "ml_features"
OUT_DIR.mkdir(parents=True, exist_ok=True)

HANDOFF_CANDIDATES = [
    ROOT / "data" / "models" / "heatgrid_ml_models_2026-06-25",
    ROOT / "_zip_read" / "model_handoff" / "heatgrid_ml_models_2026-06-25",
]
HANDOFF_ROOT = next((path for path in HANDOFF_CANDIDATES if path.exists()), None)

print("03 산출물:", WINDOW_DATASET_PATH)
print("handoff metadata 경로:", HANDOFF_ROOT)

03 산출물: C:\project3.0_2\data\processed\ml_windows\ml_window_dataset.csv
handoff metadata 경로: C:\project3.0_2\_zip_read\model_handoff\heatgrid_ml_models_2026-06-25


In [2]:
def load_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def existing_columns(df: pd.DataFrame, columns: list[str]) -> list[str]:
    return [col for col in columns if col in df.columns]


def missing_columns(df: pd.DataFrame, columns: list[str]) -> list[str]:
    return [col for col in columns if col not in df.columns]


def feature_family(column: str) -> str:
    if column.endswith(("_sin", "_cos")) or column in {
        "day_of_week", "day_of_year", "hour_of_day", "is_weekend", "is_heating_season", "month"
    }:
        return "시간/주기"
    if "__is__" in column:
        return "원핫/범주"
    if column.startswith("days_since_last") or column in {"disturbance_count", "maintenance_related"}:
        return "이벤트 이력"
    if any(token in column for token in ["__lag", "__delta1", "__roll"]):
        return "시계열 파생"
    if column in {"anomaly_score", "risk_score", "risk_probability"}:
        return "모델 결과 컨텍스트"
    return "센서 통계"


LEAKAGE_PATTERNS = [
    r"^label$",
    r"^fault_label$",
    r"^fault_event_id$",
    r"^estimated_lead_time_hours$",
    r"^split_",
    r"^use_for_supervised_training$",
    r"^window_source_type$",
    r"^window_start$",
    r"^window_end$",
    r"^source_file$",
    r"^main_missing_sensors$",
    r"^main_changed_sensors$",
    r"^normal_reference_",
    r"^leakage_",
    r"^related_",
    r"^model_explanation_",
    r"^priority_",
]


def is_leakage_or_metadata(column: str) -> bool:
    return any(re.search(pattern, column) for pattern in LEAKAGE_PATTERNS)

## 1. 03 산출물 읽기

`04`는 03 산출물을 읽기만 한다. 파일이 없으면 03을 먼저 실행해야 한다.

In [3]:
if not WINDOW_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"03 산출물이 없습니다: {WINDOW_DATASET_PATH}. "
        "PREPROCESSING/osj/03_preprocess_windows.ipynb를 먼저 실행한 뒤 04를 다시 실행하세요."
    )

df = pd.read_csv(WINDOW_DATASET_PATH)
print("행/열:", df.shape)
display(df.head())

행/열: (3270, 252)


,manufacturer,substation_id,source_file,window_start,window_end,row_count,expected_row_count,median_interval_minutes,invalid_timestamp_rows_in_file,timestamp_gap_count,...,s_hc1.1_heating_pump_status__change_count,s_hc1.3_control_unit_mode__dominant,s_hc1.3_control_unit_mode__nunique,s_hc1.3_control_unit_mode__change_count,split_time_based,split_substation_based,split_regime_based,normal_reference_outlier,normal_reference_outlier_count,normal_reference_filter_reason
0,manufacturer 1,1,substation_1.csv,2019-12-01 00:00:00,2019-12-01 06:00:00,36,36,10.0,0,0,...,NaN,NaN,NaN,NaN,validation,holdout,train,False,0,NaN
1,manufacturer 1,1,substation_1.csv,2019-12-01 06:00:00,2019-12-01 12:00:00,36,36,10.0,0,0,...,NaN,NaN,NaN,NaN,validation,holdout,train,False,0,NaN
2,manufacturer 1,1,substation_1.csv,2019-12-01 12:00:00,2019-12-01 18:00:00,36,36,10.0,0,0,...,NaN,NaN,NaN,NaN,validation,holdout,train,False,0,NaN
3,manufacturer 1,1,substation_1.csv,2019-12-01 18:00:00,2019-12-02 00:00:00,36,36,10.0,0,0,...,NaN,NaN,NaN,NaN,validation,holdout,train,False,0,NaN
4,manufacturer 1,1,substation_1.csv,2019-12-02 00:00:00,2019-12-02 06:00:00,36,36,10.0,0,0,...,NaN,NaN,NaN,NaN,validation,holdout,train,False,0,NaN


## 2. handoff metadata 읽기

Isolation Forest, Risk LightGBM, Leadtime LightGBM이 기존에 기대하던 feature 목록을 읽는다. 이 목록을 기준으로 현재 03 산출물에 실제 존재하는 컬럼만 선택한다.

In [4]:
if HANDOFF_ROOT is None:
    raise FileNotFoundError(
        "model_handoff metadata를 찾지 못했습니다. "
        "_zip_read/model_handoff/heatgrid_ml_models_2026-06-25 또는 data/models/heatgrid_ml_models_2026-06-25 경로를 확인하세요."
    )

anomaly_meta = load_json(HANDOFF_ROOT / "anomaly" / "baseline_model_metadata.json")
risk_meta = load_json(HANDOFF_ROOT / "risk" / "risk_model_metadata.json")
leadtime_meta = load_json(HANDOFF_ROOT / "leadtime" / "leadtime_bucket_model_promoted_metadata.json")

handoff_anomaly_features = anomaly_meta.get("selected_feature_columns", [])
handoff_risk_features = risk_meta.get("model_feature_columns", [])
handoff_leadtime_features = leadtime_meta.get("model_feature_columns", [])

print("Isolation Forest handoff feature:", len(handoff_anomaly_features))
print("Risk LightGBM handoff feature:", len(handoff_risk_features))
print("Leadtime LightGBM handoff feature:", len(handoff_leadtime_features))

Isolation Forest handoff feature: 195
Risk LightGBM handoff feature: 189
Leadtime LightGBM handoff feature: 221


## 3. 모델별 feature set 확정

새 feature를 만들지 않고, 현재 dataset에 존재하는 handoff feature만 남긴다. Isolation Forest는 195개를 절대 넘지 않는다.

In [5]:
metadata_candidates = [
    "manufacturer", "substation_id", "source_file", "window_start", "window_end",
    "main_missing_sensors", "main_changed_sensors", "season_bucket", "label",
    "fault_label", "fault_event_id", "estimated_lead_time_hours", "normal_event_related",
    "maintenance_related", "disturbance_count", "leakage_blocked_fault_count",
    "window_source_type", "use_for_supervised_training", "configuration_type",
    "normal_reference_group", "split_time_based", "split_substation_based", "split_regime_based",
]

metadata_columns = existing_columns(df, metadata_candidates)
anomaly_features = existing_columns(df, handoff_anomaly_features)
risk_features = existing_columns(df, handoff_risk_features)
leadtime_features = existing_columns(df, handoff_leadtime_features)

if len(anomaly_features) > 195:
    raise ValueError("Isolation Forest feature가 195개를 초과했습니다. 04 설정을 다시 확인해야 합니다.")

all_selected_features = sorted(set(anomaly_features) | set(risk_features) | set(leadtime_features))

summary = pd.DataFrame([
    {"모델": "Isolation Forest", "handoff_feature": len(handoff_anomaly_features), "현재_선택_feature": len(anomaly_features), "누락_feature": len(missing_columns(df, handoff_anomaly_features))},
    {"모델": "Risk LightGBM", "handoff_feature": len(handoff_risk_features), "현재_선택_feature": len(risk_features), "누락_feature": len(missing_columns(df, handoff_risk_features))},
    {"모델": "Leadtime LightGBM", "handoff_feature": len(handoff_leadtime_features), "현재_선택_feature": len(leadtime_features), "누락_feature": len(missing_columns(df, handoff_leadtime_features))},
])
display(summary)

,모델,handoff_feature,현재_선택_feature,누락_feature
0,Isolation Forest,195,151,44
1,Risk LightGBM,189,144,45
2,Leadtime LightGBM,221,144,77


## 4. 선택 리포트 생성

각 컬럼이 어떤 모델에서 사용되는지, 현재 dataset에 존재하는지, 결측률이 어느 정도인지 기록한다.

In [6]:
report_columns = sorted(set(df.columns) | set(handoff_anomaly_features) | set(handoff_risk_features) | set(handoff_leadtime_features))
rows = []

for col in report_columns:
    present = col in df.columns
    rows.append({
        "column": col,
        "present_in_dataset": present,
        "dtype": str(df[col].dtype) if present else None,
        "missing_rate": float(df[col].isna().mean()) if present else None,
        "nunique": int(df[col].nunique(dropna=True)) if present else None,
        "family": feature_family(col),
        "is_metadata": col in metadata_columns,
        "is_leakage_or_metadata_pattern": is_leakage_or_metadata(col),
        "anomaly_feature": col in anomaly_features,
        "risk_feature": col in risk_features,
        "leadtime_feature": col in leadtime_features,
        "missing_from_dataset": not present,
    })

feature_report = pd.DataFrame(rows)
feature_columns = feature_report[
    feature_report[["anomaly_feature", "risk_feature", "leadtime_feature"]].any(axis=1)
].copy()
metadata_df = pd.DataFrame({"column": metadata_columns})
family_summary = feature_columns.groupby("family", as_index=False).agg(
    feature_count=("column", "count"),
    avg_missing_rate=("missing_rate", "mean"),
)

missing_handoff_features = feature_report[
    feature_report["missing_from_dataset"]
    & feature_report[["anomaly_feature", "risk_feature", "leadtime_feature"]].eq(False).all(axis=1)
].copy()

display(feature_columns.head())
display(family_summary)

,column,present_in_dataset,dtype,missing_rate,nunique,family,is_metadata,is_leakage_or_metadata_pattern,anomaly_feature,risk_feature,leadtime_feature,missing_from_dataset
14,day_of_week,True,int64,0.000000,7.0,시간/주기,False,False,True,True,True,False
15,day_of_year,True,int64,0.000000,306.0,시간/주기,False,False,True,True,True,False
16,days_since_last_any_event,True,float64,0.218960,2452.0,이벤트 이력,False,False,True,True,True,False
21,days_since_last_fault_event,True,float64,0.614985,1248.0,이벤트 이력,False,False,False,True,True,False
22,days_since_last_task_event,True,float64,0.218960,2452.0,이벤트 이력,False,False,True,True,True,False


,family,feature_count,avg_missing_rate
0,센서 통계,137,0.127929
1,시간/주기,12,0.000000
2,이벤트 이력,5,0.210581


## 5. 학습용 테이블과 산출물 저장

`trainable_windows.csv`에는 metadata와 현재 선택된 feature만 남긴다. `data/processed/`는 Git 추적 대상이 아니므로 재생성 가능한 산출물로 둔다.

In [7]:
if "label" in df.columns:
    label_distribution = df.groupby("label", dropna=False).size().reset_index(name="row_count")
else:
    label_distribution = pd.DataFrame(columns=["label", "row_count"])

trainable_columns = metadata_columns + all_selected_features
trainable_windows = df[[col for col in trainable_columns if col in df.columns]].copy()

trainable_windows.to_csv(OUT_DIR / "trainable_windows.csv", index=False)
feature_columns.to_csv(OUT_DIR / "feature_columns.csv", index=False)
metadata_df.to_csv(OUT_DIR / "metadata_columns.csv", index=False)
feature_report.to_csv(OUT_DIR / "feature_selection_report.csv", index=False)
family_summary.to_csv(OUT_DIR / "feature_family_summary.csv", index=False)
label_distribution.to_csv(OUT_DIR / "label_distribution.csv", index=False)
summary.to_csv(OUT_DIR / "model_feature_count_summary.csv", index=False)

missing_rows = []
for model_name, features in [
    ("Isolation Forest", handoff_anomaly_features),
    ("Risk LightGBM", handoff_risk_features),
    ("Leadtime LightGBM", handoff_leadtime_features),
]:
    for feature in missing_columns(df, features):
        missing_rows.append({"model": model_name, "column": feature, "family": feature_family(feature)})
pd.DataFrame(missing_rows).to_csv(OUT_DIR / "missing_handoff_features.csv", index=False)

print("저장 완료")
for path in sorted(OUT_DIR.glob("*.csv")):
    print(path)

저장 완료
C:\project3.0_2\data\processed\ml_features\feature_columns.csv
C:\project3.0_2\data\processed\ml_features\feature_family_summary.csv
C:\project3.0_2\data\processed\ml_features\feature_selection_report.csv
C:\project3.0_2\data\processed\ml_features\label_distribution.csv
C:\project3.0_2\data\processed\ml_features\metadata_columns.csv
C:\project3.0_2\data\processed\ml_features\missing_handoff_features.csv
C:\project3.0_2\data\processed\ml_features\model_feature_count_summary.csv
C:\project3.0_2\data\processed\ml_features\trainable_windows.csv


## 6. 확인 기준

- Isolation Forest feature 수가 195개를 넘지 않아야 한다.
- `missing_handoff_features.csv`에서 누락 컬럼을 확인한다.
- risk와 leadtime에서 빠진 컬럼 중 `anomaly_score`, `risk_score`, `risk_probability`, lag/rolling 계열은 후속 05/06 단계에서 다룬다.